# GeoAGLA -- Student B (E3-E5) -- Full Pipeline, subset 10% MUSAN (even, seed=42)
## Ban day du nhat -- ho tro chay chia lam 2 phien (nua dau / nua sau)

**Cach dung qua 2 phien Kaggle:**

- **Phien 1:** o cell CONFIG, dat `SESSION_MAX_FILES = 101`, `RESUME_INPUT_DIR = None`.
  Chay het notebook (Save & Run All / Commit). Batch 00-50 se hoan tat va duoc luu
  lai trong Output cua version nay.
- **Giua 2 phien:** vao notebook (hoac tao version moi) -> **Add Input** -> chon
  chinh Output cua version Phien 1 vua commit.
- **Phien 2:** o cell CONFIG, dat `SESSION_MAX_FILES = None` (chay toan bo 202 file),
  `RESUME_INPUT_DIR = "/kaggle/input/<ten-output-vua-them>/results/phase2b_even10pct_seed42"`
  (sua dung ten thuc te). Chay lai toan bo notebook -- cac batch da xong (00-50)
  se duoc khoi phuc va TU DONG BI BO QUA, chi batch con thieu (51-100) moi chay moi.
  Cuoi cung script tu gop toan bo 101 batch thanh 1 summary.json duy nhat.

**Accelerator: None (CPU)** -- package chi dung NumPy/SciPy/Matplotlib, khong can GPU.


## 0. CONFIG -- CHI SUA O DAY GIUA 2 PHIEN, KHONG SUA CAC CELL KHAC

In [1]:
# ============================================================
# CONFIG -- doi 2 bien nay giua Phien 1 va Phien 2
# ============================================================

# Phien 1: 101  (nua dau, ~50% cua 202 file)
# Phien 2: None (chay toan bo 202 file -- batch da xong se tu dong bi bo qua)
SESSION_MAX_FILES = None 

# Phien 1: None
# Phien 2: duong dan toi thu muc phase2b_even10pct_seed42 trong OUTPUT cua Phien 1
#          (sau khi da "Add Input" output cua version Phien 1 vao notebook nay)
RESUME_INPUT_DIR = "/kaggle/input/datasets/ndkhoa240805/50batch/kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_seed42"
# vi du Phien 2:
# RESUME_INPUT_DIR = "/kaggle/input/geoagla-phien1-output/results/phase2b_even10pct_seed42"

# Co chay pilot 4-file kiem tra thoi gian khong (an toan de True ca 2 phien,
# chi mat vai chuc giay, giup phat hien loi moi truong som)
RUN_PILOT = True

BATCH_SIZE = 2
SUBSET_ROOT = "/kaggle/input/datasets/ndkhoa240805/musan-even-10percent/even_data"
PHASE2B_OUT_NAME = "phase2b_even10pct_seed42"

print("CONFIG:")
print(f"  SESSION_MAX_FILES = {SESSION_MAX_FILES}")
print(f"  RESUME_INPUT_DIR  = {RESUME_INPUT_DIR}")
print(f"  RUN_PILOT         = {RUN_PILOT}")

CONFIG:
  SESSION_MAX_FILES = None
  RESUME_INPUT_DIR  = /kaggle/input/datasets/ndkhoa240805/50batch/kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_seed42
  RUN_PILOT         = True


In [2]:
   !ls /kaggle/input/datasets/ndkhoa240805/50batch/kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_seed42

audio_manifest.json  _batch_12	_batch_27  _batch_42
basin_results.csv    _batch_13	_batch_28  _batch_43
basin_results.json   _batch_14	_batch_29  _batch_44
_batch_00	     _batch_15	_batch_30  _batch_45
_batch_01	     _batch_16	_batch_31  _batch_46
_batch_02	     _batch_17	_batch_32  _batch_47
_batch_03	     _batch_18	_batch_33  _batch_48
_batch_04	     _batch_19	_batch_34  _batch_49
_batch_05	     _batch_20	_batch_35  _batch_50
_batch_06	     _batch_21	_batch_36  figures
_batch_07	     _batch_22	_batch_37  geometry_results.csv
_batch_08	     _batch_23	_batch_38  geometry_results.json
_batch_09	     _batch_24	_batch_39  initialization_summary.json
_batch_10	     _batch_25	_batch_40  summary.json
_batch_11	     _batch_26	_batch_41


In [3]:
# 1. Kiem tra dataset input
!ls -la /kaggle/input/datasets/ndkhoa240805/geo-agla-phase2b-code/geometry-certified-gla-phase2b-v1
!ls -la {SUBSET_ROOT}

total 100
drwxr-xr-x  9 nobody nogroup    0 Sep 13 04:39 .
drwxr-xr-x  3 nobody nogroup    0 Sep 13 04:39 ..
-rw-r--r--  1 nobody nogroup  333 Sep 13 04:39 CITATION.cff
drwxr-xr-x  2 nobody nogroup    0 Sep 13 04:39 configs
drwxr-xr-x  2 nobody nogroup    0 Sep 13 04:39 data
drwxr-xr-x 11 nobody nogroup    0 Sep 13 04:39 experiments
-rw-r--r--  1 nobody nogroup 1056 Sep 13 04:39 LICENSE
-rw-r--r--  1 nobody nogroup 1569 Sep 13 04:39 PHASE1B_SMOKE_REPORT.md
-rw-r--r--  1 nobody nogroup 1782 Sep 13 04:39 PHASE1C_README.md
-rw-r--r--  1 nobody nogroup 3700 Sep 13 04:39 PHASE1C_VERIFICATION_REPORT.md
-rw-r--r--  1 nobody nogroup 1700 Sep 13 04:39 PHASE1D_README.md
-rw-r--r--  1 nobody nogroup 1240 Sep 13 04:39 Phase-1D_Reproduction_Guide_v1.md
-rw-r--r--  1 nobody nogroup 4226 Sep 13 04:39 PHASE1D_VERIFICATION_REPORT.md
-rw-r--r--  1 nobody nogroup  471 Sep 13 04:39 PHASE1E_README.md
-rw-r--r--  1 nobody nogroup 1689 Sep 13 04:39 Phase-1E_Reproduction_Guide_v1.md
-rw-r--r--  1 nobody nogro

In [4]:
# 2. Copy code sang vung ghi duoc
import shutil, os

SRC_CODE = "/kaggle/input/datasets/ndkhoa240805/geo-agla-phase2b-code/geometry-certified-gla-phase2b-v1"
PKG_DIR = "/kaggle/working/geo-agla-phase2b"

if not os.path.exists(PKG_DIR):
    shutil.copytree(SRC_CODE, PKG_DIR)
%cd /kaggle/working/geo-agla-phase2b

/kaggle/working/geo-agla-phase2b


In [5]:
# 3. Cai dat package
!pip install -e . -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for geometry-certified-gla (pyproject.toml) ... done


In [6]:
# 4. Xac nhan moi truong -- ky vong 17 passed. NEU FAIL: DUNG LAI, khong chay tiep.
import subprocess
res = subprocess.run(["pytest", "-q"], capture_output=True, text=True)
print(res.stdout[-2000:])
if res.returncode != 0:
    print(res.stderr[-2000:])
    raise RuntimeError("pytest FAILED -- dung lai, kiem tra moi truong truoc khi tiep tuc.")

attern.py:64
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
    parse = parser.parseString(pattern)

../../../usr/local/lib/python3.12/dist-packages/mat

In [7]:
# 5. Ghi lai moi truong Kaggle
import subprocess
from pathlib import Path

SR = Path("/kaggle/working/student_runs/student_B")
(SR / "terminal_logs").mkdir(parents=True, exist_ok=True)

env_txt = subprocess.run(["python", "--version"], capture_output=True, text=True).stdout
env_txt += subprocess.run(["python", "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(SR / "environment_kaggle_subset10pct.txt").write_text(env_txt)
print(env_txt[:300])

Python 3.12.13
a2a-sdk==0.3.26
absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==22.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.5
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotat


## B0 -- E3: Passive Ritz Estimation (khong can audio)

In [8]:
# 6. Chay B0 -- Phase-1D
res = get_ipython().getoutput("cd /kaggle/working/geo-agla-phase2b && python -m reproduce.run_phase1d")
print("\n".join(res[-40:]))

Phase-1D case 16: g24_w8_h4_f8 seed=1 eps=0.0001
Phase-1D case 17: g24_w8_h4_f8 seed=2 eps=0.001
Phase-1D case 18: g24_w8_h4_f8 seed=2 eps=0.0001
Phase-1D case 19: h24_w8_h4_f8 seed=0 eps=0.001
Phase-1D case 20: h24_w8_h4_f8 seed=0 eps=0.0001
Phase-1D case 21: h24_w8_h4_f8 seed=1 eps=0.001
Phase-1D case 22: h24_w8_h4_f8 seed=1 eps=0.0001
Phase-1D case 23: h24_w8_h4_f8 seed=2 eps=0.001
Phase-1D case 24: h24_w8_h4_f8 seed=2 eps=0.0001
Phase-1D case 25: g32_w16_h4_f16 seed=0 eps=0.001
Phase-1D case 26: g32_w16_h4_f16 seed=0 eps=0.0001
Phase-1D case 27: g32_w16_h4_f16 seed=1 eps=0.001
Phase-1D case 28: g32_w16_h4_f16 seed=1 eps=0.0001
Phase-1D case 29: g32_w16_h4_f16 seed=2 eps=0.001
Phase-1D case 30: g32_w16_h4_f16 seed=2 eps=0.0001
Phase-1D case 31: h32_w16_h4_f16 seed=0 eps=0.001
Phase-1D case 32: h32_w16_h4_f16 seed=0 eps=0.0001
Phase-1D case 33: h32_w16_h4_f16 seed=1 eps=0.001
Phase-1D case 34: h32_w16_h4_f16 seed=1 eps=0.0001
Phase-1D case 35: h32_w16_h4_f16 seed=2 eps=0.001
Phase-1D

In [9]:
# 7. Copy ket qua Phase-1D
import shutil
from pathlib import Path
SR = Path("/kaggle/working/student_runs/student_B")
src = Path("/kaggle/working/geo-agla-phase2b/results/phase1d")
if src.exists():
    shutil.copytree(src, SR / "phase1d", dirs_exist_ok=True)
    print("OK:", (SR / "phase1d" / "summary.json").read_text()[:300])
else:
    print("CANH BAO: results/phase1d khong ton tai!")

OK: {
  "d0_max_abs_error": 1.1601830607332886e-13,
  "n_stft_estimator_rows": 108,
  "selected_memory": 3,
  "selected_median_rel_error": 0.0013147740132248304,
  "selected_p90_rel_error": 0.004994520259143865,
  "selected_median_under_fraction": 0.990909090909091,
  "identifiability_trajectory_max_dif


## B1 -- E4: Online-lambda+ Rate Realization (khong can audio)

Dung `python -m reproduce.run_phase1er` de tranh loi dong goi da biet
("Phase-1E-R packaging defect") trong chinh code goc.

In [10]:
# 8. Chay B1 -- Phase-1E-R
res = get_ipython().getoutput("cd /kaggle/working/geo-agla-phase2b && python -m reproduce.run_phase1er")
print("\n".join(res[-40:]))

  },
  "H4_performance_pass": true,
  "H5_eventual_inactivity_pass": true,
  "E1_safe_fallback_pass": true,
  "phase1er_gate": "GREEN"
}
[
  {
    "n_iter": 120,
    "n": 24,
    "beat_fraction": 0.3333333333333333,
    "median_ratio": 1.0067527817511484,
    "p90_ratio": 1.0165880669460576,
    "median_final_edge_rel_error": 5.730935443362452e-05
  },
  {
    "n_iter": 180,
    "n": 24,
    "beat_fraction": 0.5,
    "median_ratio": 0.9992602339901818,
    "p90_ratio": 1.00562488849491,
    "median_final_edge_rel_error": 2.9177805666761604e-06
  },
  {
    "n_iter": 240,
    "n": 24,
    "beat_fraction": 0.75,
    "median_ratio": 0.9975016364314921,
    "p90_ratio": 1.0028310057929517,
    "median_final_edge_rel_error": 9.921684914589217e-06
  },
  {
    "n_iter": 360,
    "n": 24,
    "beat_fraction": 0.9583333333333334,
    "median_ratio": 0.9946815812419686,
    "p90_ratio": 0.9987164777476231,
    "median_final_edge_rel_error": 2.0516591810371757e-05
  }
]


In [11]:
# 9. Copy ket qua Phase-1E-R
import shutil
from pathlib import Path
SR = Path("/kaggle/working/student_runs/student_B")
src = Path("/kaggle/working/geo-agla-phase2b/results/phase1er")
if src.exists():
    shutil.copytree(src, SR / "phase1er", dirs_exist_ok=True)
    print("OK:", (SR / "phase1er" / "summary.json").read_text()[:300])
else:
    print("CANH BAO: results/phase1er khong ton tai!")

OK: {
  "n_runs": 192,
  "selected_repair": {
    "policy": "dynamic_envelope",
    "n": 24,
    "beat_fraction": 0.75,
    "median_q_ratio_fgla": 0.9975016364314921,
    "p90_q_ratio_fgla": 1.0028310057929517,
    "median_q_ratio_oracle": 1.004104869941964,
    "mean_late_reject_fraction": 0.0,
    "me


## B2 -- E5a: Signal-Scale Transfer (tin hieu tong hop, khong can audio)

In [12]:
# 10. Chay B2 -- Phase-2A
res = get_ipython().getoutput("cd /kaggle/working/geo-agla-phase2b && python -m reproduce.run_phase2a")
print("\n".join(res[-40:]))

Phase-2A harmonic_n256 lambda+ 0.999358225069577 geom err 0.00012902871982086958 time 2.18 s
Phase-2A chirp_n512 lambda+ 0.9993605257987934 geom err 0.00018343343047535988 time 4.31 s
Phase-2A speechlike_n512 lambda+ 0.9975869378254596 geom err 0.0008779577104445768 time 3.07 s
Phase-2A transient_n1024 lambda+ 0.999962991639193 geom err 0.0003589520741435361 time 7.1 s
{
  "profile": "controlled-signal-scale",
  "n_cases_total": 4,
  "n_regular_runs": 4,
  "n_irregular_skipped": 0,
  "max_geometry_relative_error": 0.0008779577104445768,
  "median_final_edge_relative_error": 8.41024082524648e-06,
  "online_beat_fraction_vs_fgla": 0.75,
  "median_q_ratio_online_vs_fgla": 0.99461084416267,
  "p90_q_ratio_online_vs_fgla": 1.0013038433828771,
  "mean_late_reject_fraction": 0.0,
  "fallback_e1_failures": 0,
  "G2A_geometry_transfer_pass": true,
  "G2B_active_edge_tracking_pass": true,
  "G2C_online_rate_transfer_pass": true,
  "G2D_safeguard_transfer_pass": true,
  "phase2a_controlled_gate":

In [13]:
# 11. Copy ket qua Phase-2A
import shutil
from pathlib import Path
SR = Path("/kaggle/working/student_runs/student_B")
src = Path("/kaggle/working/geo-agla-phase2b/results/phase2a")
if src.exists():
    shutil.copytree(src, SR / "phase2a", dirs_exist_ok=True)
    print("OK:", (SR / "phase2a" / "summary.json").read_text()[:300])
else:
    print("CANH BAO: results/phase2a khong ton tai!")

OK: {
  "profile": "controlled-signal-scale",
  "n_cases_total": 4,
  "n_regular_runs": 4,
  "n_irregular_skipped": 0,
  "max_geometry_relative_error": 0.0008779577104445768,
  "median_final_edge_relative_error": 8.41024082524648e-06,
  "online_beat_fraction_vs_fgla": 0.75,
  "median_q_ratio_online_vs_f


In [14]:
%%writefile reproduce/run_phase2b_checkpointed.py
#!/usr/bin/env python3
"""
run_phase2b_checkpointed.py
============================
Wrapper quanh `experiments.phase2b_real_audio.run.run()` để CHIA NHỎ danh sách
file audio thành nhiều "batch" độc lập — mỗi batch tự ghi kết quả xuống đĩa
ngay khi xong — nhằm đảm bảo KHÔNG MẤT kết quả nếu phiên chạy (Kaggle, giới
hạn ~12h, hoặc bất kỳ máy nào) bị ngắt giữa chừng.

KHÔNG có dòng nào trong experiments/phase2b_real_audio/run.py bị sửa.
Script này chỉ IMPORT và GỌI LẠI đúng hàm run() gốc cho từng batch nhỏ, sau đó
gộp (merge) các JSON/CSV bằng ĐÚNG công thức gate/summary/figure đã có sẵn
trong run.py (copy y nguyên phần tổng hợp cuối — không đổi bất kỳ công thức
khoa học nào, chỉ áp dụng nó lên dữ liệu đã gộp từ nhiều batch thay vì một
lần chạy độc khối).

Cơ chế resume: mỗi batch ghi `out/_batch_NN/summary.json` khi hoàn tất. Lần
chạy lại (kể cả ở phiên Kaggle hoàn toàn mới, miễn giữ lại thư mục `out/`)
sẽ tự động BỎ QUA các batch đã có summary.json và chỉ chạy tiếp các batch
còn thiếu.

Cách dùng:
    # Preflight nhỏ (khuyến nghị chạy trước để đo thời gian/file)
    python -m reproduce.run_phase2b_checkpointed \\
        --audio-dir data/my_audio --max-files 2 --batch-size 2 \\
        --out results/phase2b_preflight

    # Chạy chính thức, chia 2 file/batch -> 3 batch cho 6 file
    python -m reproduce.run_phase2b_checkpointed \\
        --audio-dir data/my_audio --max-files 6 --batch-size 2 \\
        --out results/phase2b

    # Bị ngắt giữa chừng? Chạy lại y nguyên lệnh trên -> batch đã xong tự skip
"""
from __future__ import annotations

import argparse
import csv
import json
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1]
for _p in (ROOT, ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from experiments.phase2b_real_audio.run import run as run_phase2b_batch  # noqa: E402


# ---------------------------------------------------------------------------
# 1. Liệt kê + chia batch (deterministic — cùng logic sort/slice như run.py)
# ---------------------------------------------------------------------------

def discover_wav_files(audio_dir: Path, max_files):
    paths = sorted(
        p for p in Path(audio_dir).rglob("*")
        if p.is_file() and p.suffix.lower() == ".wav"
    )
    if max_files is not None:
        paths = paths[: int(max_files)]
    return paths


def chunk(paths, batch_size):
    return [paths[i:i + batch_size] for i in range(0, len(paths), batch_size)]


# ---------------------------------------------------------------------------
# 2. Chạy từng batch qua đúng hàm run() gốc — có khả năng resume
# ---------------------------------------------------------------------------

def run_batches(paths, config_path, out_root: Path, batch_size: int):
    batches = chunk(paths, batch_size)
    batch_out_dirs = []
    for i, batch_files in enumerate(batches):
        batch_out = out_root / f"_batch_{i:02d}"
        batch_out_dirs.append(batch_out)
        marker = batch_out / "summary.json"
        if marker.exists():
            print(f"[batch {i+1}/{len(batches)}] Đã hoàn tất trước đó — bỏ qua ({marker}).")
            continue

        batch_audio_dir = out_root / f"_audio_batch_{i:02d}"
        if batch_audio_dir.exists():
            shutil.rmtree(batch_audio_dir)
        batch_audio_dir.mkdir(parents=True, exist_ok=True)
        for f in batch_files:
            shutil.copy2(f, batch_audio_dir / f.name)

        names = ", ".join(f.name for f in batch_files)
        print(f"[batch {i+1}/{len(batches)}] Đang chạy ({names}) ...", flush=True)
        t0 = time.time()
        run_phase2b_batch(str(batch_audio_dir), config_path, str(batch_out), max_files=len(batch_files))
        print(f"[batch {i+1}/{len(batches)}] Xong sau {time.time()-t0:.1f}s. Đã ghi: {marker}", flush=True)
    return batch_out_dirs


# ---------------------------------------------------------------------------
# 3. Gộp kết quả — công thức gate/summary/figure GIỮ NGUYÊN như run.py
#    (copy verbatim từ dòng "gok=..." trở xuống trong run.py, không đổi logic)
# ---------------------------------------------------------------------------

def _load(path: Path, default):
    return json.loads(path.read_text()) if path.exists() else default


def merge(batch_out_dirs, config_path: str, out_root: Path):
    C = json.loads(Path(config_path).read_text())
    manifest, geometry, basin = [], [], []
    for bd in batch_out_dirs:
        manifest.extend(_load(bd / "audio_manifest.json", []))
        geometry.extend(_load(bd / "geometry_results.json", []))
        basin.extend(_load(bd / "basin_results.json", []))

    out = out_root
    (out / "figures").mkdir(parents=True, exist_ok=True)
    (out / "audio_manifest.json").write_text(json.dumps(manifest, indent=2))
    (out / "geometry_results.json").write_text(json.dumps(geometry, indent=2, allow_nan=True))
    (out / "basin_results.json").write_text(json.dumps(basin, indent=2, allow_nan=True))
    for name, rows in [("geometry_results.csv", geometry), ("basin_results.csv", basin)]:
        if rows:
            with (out / name).open("w", newline="") as f:
                fields = sorted(set().union(*(r.keys() for r in rows)))
                w = csv.DictWriter(f, fieldnames=fields)
                w.writeheader(); w.writerows(rows)

    gok = [r for r in geometry if r.get("status") == "OK"]
    if not gok:
        summary = {"phase2b_gate": "RED", "reason": "no regular real-audio geometry cases", "n_files": len(manifest)}
        (out / "summary.json").write_text(json.dumps(summary, indent=2))
        print(json.dumps(summary, indent=2))
        return summary

    paired = [r for r in basin if r["success_online"] and r["success_fgla"] and r["q_ratio_online_fgla"] is not None]
    small = [r for r in basin if (r["init_kind"] == "transverse" and r["level"] <= 0.01)
             or (r["init_kind"] == "phase_jitter" and r["level"] <= 0.15)]
    ratios = np.asarray([r["q_ratio_online_fgla"] for r in paired], float) if paired else np.asarray([])
    edges = np.asarray([r["final_edge_rel_error"] for r in basin if r["final_edge_rel_error"] is not None], float)
    G = C["gates"]
    geom_max = float(np.max([r["geometry_rel_error"] for r in gok]))
    edge_med = float(np.median(edges)) if edges.size else float("inf")
    small_succ = float(np.mean([r["success_online"] for r in small])) if small else 0.0
    on_succ = float(np.mean([r["success_online"] for r in basin])) if basin else 0.0
    fg_succ = float(np.mean([r["success_fgla"] for r in basin])) if basin else 0.0
    beat = float(np.mean([r["beats_fgla"] for r in paired])) if paired else 0.0
    medratio = float(np.median(ratios)) if ratios.size else float("inf")
    late = float(np.mean([r["late_reject_fraction"] for r in basin])) if basin else 1.0
    e1 = int(sum(r["fallback_e1_failures"] for r in basin))
    passes = {
        "G2B_A_real_audio_geometry_pass": bool(geom_max <= G["geometry_relative_error_max"]),
        "G2B_B_active_edge_tracking_pass": bool(edge_med <= G["edge_relative_error_median_max"]),
        "G2B_C_small_basin_pass": bool(small_succ >= G["small_basin_success_fraction_min"]),
        "G2B_D_basin_nonshrink_pass": bool(on_succ + G["online_basin_deficit_vs_fgla_max"] >= fg_succ),
        "G2B_E_paired_rate_pass": bool(len(paired) > 0 and beat >= G["paired_success_online_beat_fraction_min"]
                                        and medratio < G["paired_success_median_q_ratio_max"]),
        "G2B_F_safeguard_pass": bool(late <= G["late_reject_fraction_mean_max"] and e1 <= G["fallback_e1_failures_max"]),
    }
    summary = {
        "profile": C["profile"], "n_files": len(manifest), "n_geometry_cases": len(gok),
        "n_basin_runs": len(basin), "n_paired_success_runs": len(paired),
        "max_geometry_relative_error": geom_max, "median_final_edge_relative_error": edge_med,
        "small_basin_online_success_fraction": small_succ, "overall_online_success_fraction": on_succ,
        "overall_fgla_success_fraction": fg_succ, "paired_success_online_beat_fraction": beat,
        "paired_success_median_q_ratio": medratio,
        "paired_success_p90_q_ratio": float(np.percentile(ratios, 90)) if ratios.size else None,
        "mean_late_reject_fraction": late, "fallback_e1_failures": e1,
        "near_phase_degenerate_fraction": float(np.mean([r["lambda_plus"] >= 0.9999 for r in gok])),
        **passes,
    }
    summary["phase2b_gate"] = "GREEN" if all(passes.values()) else "YELLOW"
    summary["merged_from_batches"] = len(batch_out_dirs)
    (out / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=True))

    groups = []
    for kind in sorted(set(r["init_kind"] for r in basin)):
        for level in sorted(set(r["level"] for r in basin if r["init_kind"] == kind)):
            z = [r for r in basin if r["init_kind"] == kind and r["level"] == level]
            pair = [r for r in z if r["success_online"] and r["success_fgla"] and r["q_ratio_online_fgla"] is not None]
            groups.append({
                "init_kind": kind, "level": level, "n": len(z),
                "online_success_fraction": float(np.mean([r["success_online"] for r in z])),
                "fgla_success_fraction": float(np.mean([r["success_fgla"] for r in z])),
                "paired_beat_fraction": float(np.mean([r["beats_fgla"] for r in pair])) if pair else None,
                "paired_median_q_ratio": float(np.median([r["q_ratio_online_fgla"] for r in pair])) if pair else None,
            })
    (out / "initialization_summary.json").write_text(json.dumps(groups, indent=2, allow_nan=True))

    labels = [f"{r['file']}\n{r['profile']}" for r in gok]
    x = np.arange(len(gok))
    fig, ax = plt.subplots(figsize=(max(8, 1.1 * len(gok)), 4.8))
    ax.bar(x, [r["geometry_rel_error"] for r in gok]); ax.axhline(G["geometry_relative_error_max"], linestyle="--")
    ax.set_yscale("log"); ax.set_xticks(x, labels, rotation=35, ha="right"); ax.set_ylabel("relative error")
    ax.set_title("2B-D1 — Real-audio tangent geometry transfer (merged batches)")
    fig.tight_layout(); fig.savefig(out / "figures" / "2B_D1_real_audio_geometry.png", dpi=180); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for kind in sorted(set(g["init_kind"] for g in groups)):
        z = [g for g in groups if g["init_kind"] == kind]
        ax.plot([g["level"] for g in z], [g["online_success_fraction"] for g in z], marker="o", label=f"{kind} online")
        ax.plot([g["level"] for g in z], [g["fgla_success_fraction"] for g in z], marker="x", label=f"{kind} FGLA")
    ax.set_xscale("log"); ax.set_ylim(-.02, 1.02); ax.set_xlabel("initialization level"); ax.set_ylabel("basin success fraction")
    ax.set_title("2B-D2 — Real-audio basin profile (merged batches)"); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out / "figures" / "2B_D2_basin_profile.png", dpi=180); plt.close(fig)

    if paired:
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        ax.hist([r["q_ratio_online_fgla"] for r in paired], bins=min(12, max(4, len(paired) // 2)))
        ax.axvline(1.0, linestyle="--"); ax.set_xlabel("q_online / q_FGLA"); ax.set_ylabel("count")
        ax.set_title("2B-D3 — Paired-success local rate transfer (merged batches)")
        fig.tight_layout(); fig.savefig(out / "figures" / "2B_D3_rate_ratio.png", dpi=180); plt.close(fig)

    print(f"Đã gộp {len(batch_out_dirs)} batch -> {out}")
    print(json.dumps(summary, indent=2))
    return summary


def main():
    ap = argparse.ArgumentParser(
        description="Phase-2B real-audio runner có checkpoint theo batch file audio; "
                    "dùng lại nguyên hàm run() gốc, không sửa code khoa học."
    )
    ap.add_argument("--audio-dir", required=True)
    ap.add_argument("--config", default="configs/phase2b_grid.json")
    ap.add_argument("--out", default="results/phase2b")
    ap.add_argument("--max-files", type=int, default=None)
    ap.add_argument("--batch-size", type=int, default=2, help="Số file audio xử lý mỗi batch (mặc định 2).")
    args = ap.parse_args()

    out_root = Path(args.out)
    out_root.mkdir(parents=True, exist_ok=True)

    all_paths = discover_wav_files(Path(args.audio_dir), args.max_files)
    if not all_paths:
        raise FileNotFoundError(f"Không tìm thấy file .wav nào trong {args.audio_dir}")
    n_batches = ((len(all_paths) - 1) // args.batch_size) + 1
    print(f"Tìm thấy {len(all_paths)} file WAV -> chia thành {n_batches} batch (batch-size={args.batch_size}).")

    batch_out_dirs = run_batches(all_paths, args.config, out_root, args.batch_size)
    merge(batch_out_dirs, args.config, out_root)


if __name__ == "__main__":
    main()


Writing reproduce/run_phase2b_checkpointed.py


## B3 -- E5b: Real-Audio Transfer tren subset 10% MUSAN (even, seed=42)

### Buoc 1 -- Dem so file thuc te trong subset da mount

In [15]:
# 13. Dem file .wav trong subset
from pathlib import Path

subset_files = sorted(p for p in Path(SUBSET_ROOT).rglob("*.wav") if p.is_file())
N_SUBSET = len(subset_files)
print(f"Tong so file .wav trong subset da mount: {N_SUBSET}")
print("5 file dau tien:")
for f in subset_files[:5]:
    print(" -", f.relative_to(SUBSET_ROOT))

Tong so file .wav trong subset da mount: 202
5 file dau tien:
 - musan_even_10pct_seed42/musan/music/fma/music-fma-0010.wav
 - musan_even_10pct_seed42/musan/music/fma/music-fma-0018.wav
 - musan_even_10pct_seed42/musan/music/fma/music-fma-0024.wav
 - musan_even_10pct_seed42/musan/music/fma/music-fma-0032.wav
 - musan_even_10pct_seed42/musan/music/fma/music-fma-0042.wav


### Buoc 2 -- Khoi phuc batch da hoan tat tu phien truoc (neu RESUME_INPUT_DIR duoc dat)

In [16]:
# 14. Resume bridge -- copy cac batch da xong tu output phien truoc (neu co)
import shutil
from pathlib import Path

CURRENT_OUT = Path("/kaggle/working/geo-agla-phase2b/results") / PHASE2B_OUT_NAME
CURRENT_OUT.mkdir(parents=True, exist_ok=True)

if RESUME_INPUT_DIR:
    prev = Path(RESUME_INPUT_DIR)
    if not prev.exists():
        raise FileNotFoundError(
            f"RESUME_INPUT_DIR khong ton tai: {prev} -- kiem tra lai ban da "
            f"'Add Input' dung output cua phien truoc chua, va sua duong dan cho khop."
        )
    restored = 0
    for batch_dir in sorted(prev.glob("_batch_*")):
        if (batch_dir / "summary.json").exists():
            shutil.copytree(batch_dir, CURRENT_OUT / batch_dir.name, dirs_exist_ok=True)
            restored += 1
            print(f"Da khoi phuc: {batch_dir.name}")
    print(f"\nTong cong khoi phuc {restored} batch tu phien truoc.")
else:
    print("RESUME_INPUT_DIR = None -- day la phien dau tien, khong co gi de khoi phuc.")

Da khoi phuc: _batch_00
Da khoi phuc: _batch_01
Da khoi phuc: _batch_02
Da khoi phuc: _batch_03
Da khoi phuc: _batch_04
Da khoi phuc: _batch_05
Da khoi phuc: _batch_06
Da khoi phuc: _batch_07
Da khoi phuc: _batch_08
Da khoi phuc: _batch_09
Da khoi phuc: _batch_10
Da khoi phuc: _batch_11
Da khoi phuc: _batch_12
Da khoi phuc: _batch_13
Da khoi phuc: _batch_14
Da khoi phuc: _batch_15
Da khoi phuc: _batch_16
Da khoi phuc: _batch_17
Da khoi phuc: _batch_18
Da khoi phuc: _batch_19
Da khoi phuc: _batch_20
Da khoi phuc: _batch_21
Da khoi phuc: _batch_22
Da khoi phuc: _batch_23
Da khoi phuc: _batch_24
Da khoi phuc: _batch_25
Da khoi phuc: _batch_26
Da khoi phuc: _batch_27
Da khoi phuc: _batch_28
Da khoi phuc: _batch_29
Da khoi phuc: _batch_30
Da khoi phuc: _batch_31
Da khoi phuc: _batch_32
Da khoi phuc: _batch_33
Da khoi phuc: _batch_34
Da khoi phuc: _batch_35
Da khoi phuc: _batch_36
Da khoi phuc: _batch_37
Da khoi phuc: _batch_38
Da khoi phuc: _batch_39
Da khoi phuc: _batch_40
Da khoi phuc: _b

### Buoc 3 -- Pilot 4 file (co kiem tra loi ro rang -- KHONG bao gio bao 'OK' neu lenh that bai)

In [17]:
# 15. Pilot -- neu that bai se DUNG NOTEBOOK LAI, khong chay tiep buoc sau
import time, subprocess

if RUN_PILOT:
    t0 = time.time()
    result = subprocess.run(
        ["python", "-m", "reproduce.run_phase2b_checkpointed",
         "--audio-dir", SUBSET_ROOT,
         "--max-files", "4", "--batch-size", "2", "--out", "results/phase2b_pilot"],
        capture_output=True, text=True,
        cwd="/kaggle/working/geo-agla-phase2b",
    )
    elapsed = time.time() - t0

    if result.returncode != 0:
        print("!!! PILOT THAT BAI -- KHONG duoc chay buoc full tiep theo !!!")
        print("--- STDOUT ---\n", result.stdout[-2000:])
        print("--- STDERR ---\n", result.stderr[-2000:])
        raise RuntimeError("Pilot run failed -- fix loi truoc khi tiep tuc.")

    per_file = elapsed / 4
    remaining = N_SUBSET if SESSION_MAX_FILES is None else SESSION_MAX_FILES
    eta_sec = per_file * remaining
    print(f"Pilot THANH CONG. Thoi gian: {elapsed:.1f}s -> {per_file:.1f}s/file")
    print(f"UOC LUONG thoi gian cho phien nay ({remaining} file muc tieu): {eta_sec/3600:.2f} gio")
    if eta_sec > 9 * 3600:
        print("!!! CANH BAO: uoc luong vuot 9h -- can giam SESSION_MAX_FILES o CONFIG.")
    else:
        print("OK -- an toan chay full trong phien nay.")
else:
    print("RUN_PILOT = False -- bo qua buoc do thoi gian.")

Pilot THANH CONG. Thoi gian: 1127.1s -> 281.8s/file
UOC LUONG thoi gian cho phien nay (202 file muc tieu): 15.81 gio
!!! CANH BAO: uoc luong vuot 9h -- can giam SESSION_MAX_FILES o CONFIG.


### Buoc 4 -- Chay chinh thuc (theo SESSION_MAX_FILES da dat o CONFIG)

In [18]:
# 16. Chay chinh thuc Phase-2B checkpointed
import subprocess

cmd = [
    "python", "-m", "reproduce.run_phase2b_checkpointed",
    "--audio-dir", SUBSET_ROOT,
    "--batch-size", str(BATCH_SIZE),
    "--out", f"results/{PHASE2B_OUT_NAME}",
]
if SESSION_MAX_FILES is not None:
    cmd += ["--max-files", str(SESSION_MAX_FILES)]

print("Lenh chay:", " ".join(cmd))
result = subprocess.run(cmd, cwd="/kaggle/working/geo-agla-phase2b")
if result.returncode != 0:
    raise RuntimeError("Phase-2B run THAT BAI -- kiem tra log truoc khi tiep tuc.")
print("Phase-2B run hoan tat (hoac dang cho phien tiep theo neu chua du file).")

Lenh chay: python -m reproduce.run_phase2b_checkpointed --audio-dir /kaggle/input/datasets/ndkhoa240805/musan-even-10percent/even_data --batch-size 2 --out results/phase2b_even10pct_seed42
Tìm thấy 202 file WAV -> chia thành 101 batch (batch-size=2).
[batch 1/101] Đã hoàn tất trước đó — bỏ qua (results/phase2b_even10pct_seed42/_batch_00/summary.json).
[batch 2/101] Đã hoàn tất trước đó — bỏ qua (results/phase2b_even10pct_seed42/_batch_01/summary.json).
[batch 3/101] Đã hoàn tất trước đó — bỏ qua (results/phase2b_even10pct_seed42/_batch_02/summary.json).
[batch 4/101] Đã hoàn tất trước đó — bỏ qua (results/phase2b_even10pct_seed42/_batch_03/summary.json).
[batch 5/101] Đã hoàn tất trước đó — bỏ qua (results/phase2b_even10pct_seed42/_batch_04/summary.json).
[batch 6/101] Đã hoàn tất trước đó — bỏ qua (results/phase2b_even10pct_seed42/_batch_05/summary.json).
[batch 7/101] Đã hoàn tất trước đó — bỏ qua (results/phase2b_even10pct_seed42/_batch_06/summary.json).
[batch 8/101] Đã hoàn tất tr

In [19]:
# 17. Copy ket qua Phase-2B vao student_runs + zip rieng (loai tru batch trung gian)
import shutil
from pathlib import Path

SR = Path("/kaggle/working/student_runs/student_B")
src = Path("/kaggle/working/geo-agla-phase2b/results") / PHASE2B_OUT_NAME
dst = SR / PHASE2B_OUT_NAME
if src.exists() and (src / "summary.json").exists():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("OK:", (dst / "summary.json").read_text()[:500])
else:
    print(f"CANH BAO: {src}/summary.json chua ton tai -- binh thuong neu day la phien"
          f" giua chung (SESSION_MAX_FILES < tong so file), merge chi xay ra khi"
          f" du toan bo batch.")

get_ipython().system(
    f"zip -r /kaggle/working/phase2b_{PHASE2B_OUT_NAME}_results.zip "
    f"/kaggle/working/geo-agla-phase2b/results/{PHASE2B_OUT_NAME} -x '*/_audio_batch_*/*'"
)
print("Da nen ket qua Phase-2B (batch hien co) rieng, phong ho truoc khi commit.")

OK: {
  "profile": "real-audio-basin-geometry",
  "n_files": 201,
  "n_geometry_cases": 390,
  "n_basin_runs": 4680,
  "n_paired_success_runs": 2541,
  "max_geometry_relative_error": 0.002532204941629028,
  "median_final_edge_relative_error": 6.266596416415596e-05,
  "small_basin_online_success_fraction": 0.8224358974358974,
  "overall_online_success_fraction": 0.7493589743589744,
  "overall_fgla_success_fraction": 0.5487179487179488,
  "paired_success_online_beat_fraction": 0.3447461629279811,
  "p
  adding: kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_seed42/ (stored 0%)
  adding: kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_seed42/_batch_70/ (stored 0%)
  adding: kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_seed42/_batch_70/geometry_results.csv (deflated 48%)
  adding: kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_seed42/_batch_70/basin_results.json (deflated 83%)
  adding: kaggle/working/geo-agla-phase2b/results/phase2b_even10pct_s

## Cross-check tai lap -- Phase-1C (phan viec Student A)

**Luu y:** chi can chay 1 lan (khong can lap lai o phien 2 neu da co ket qua GREEN
tu phien 1) -- Phase-1C khong lien quan den subset audio, ket qua se giong het.

In [20]:
# 18. Chay cross-check Phase-1C
import subprocess
from pathlib import Path

SR = Path("/kaggle/working/student_runs/student_B")
(SR / "phase1c").mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    ["python", "-m", "reproduce.run_phase1c"],
    capture_output=True, text=True,
    cwd="/kaggle/working/geo-agla-phase2b",
)
log_content = (
    "Command: python -m reproduce.run_phase1c\n"
    "Environment: Kaggle Notebook, CPU-only\n\n"
    "--- STDOUT ---\n" + result.stdout +
    "\n--- STDERR ---\n" + result.stderr
)
(SR / "terminal_logs" / "phase1c_crosscheck_command.txt").write_text(log_content)
print(result.stdout[-3000:])

/matplotlib/_fontconfig_pattern.py:89
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
    parser.resetCache()

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45
  /usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
    ParserElement.enablePackrat()

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
17 passed, 14 warnings in 2.54s

+ /usr/bin/python3 -m experiments.phase1c_multi.run --config configs/phase1c_gri

In [21]:
# 19. Copy ket qua Phase-1C
import shutil
from pathlib import Path
src = Path("/kaggle/working/geo-agla-phase2b/results/phase1c")
dst = Path("/kaggle/working/student_runs/student_B/phase1c")
if src.exists():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print((dst / "summary.json").read_text())
else:
    print("CANH BAO: results/phase1c khong ton tai!")

{
  "n_h1_cases": 30,
  "n_regular_h2_cases": 30,
  "n_h2_runs": 270,
  "h1_max_relative_spectral_error": 1.2263698795590144e-15,
  "h1_median_relative_spectral_error": 7.9966565662874625e-16,
  "h1_all_single_neutral": true,
  "h1_pass": true,
  "gla_median_relative_rate_error": 1.2286230648297005e-08,
  "gla_p90_relative_rate_error": 3.6789248966891846e-07,
  "fgla_opt_median_relative_rate_error": 0.005569595922637191,
  "fgla_opt_p90_relative_rate_error": 0.00931552665891647,
  "oracle_median_relative_rate_error": 0.005967580869544625,
  "oracle_p90_relative_rate_error": 0.012951107724058587,
  "observed_ordering_fraction": 1.0,
  "predicted_ordering_fraction": 1.0,
  "h2_ordering_pass": true,
  "phase1c_gate": "GREEN"
}


## Kiem tra cuoi cung truoc khi dong goi

In [22]:
# 20. Kiem tra cuoi -- liet ke day du, phan biet ro trang thai "dang cho" vs "thieu that"
from pathlib import Path

SR = Path("/kaggle/working/student_runs/student_B")
expected = [
    "environment_kaggle_subset10pct.txt",
    "phase1d/summary.json",
    "phase1er/summary.json",
    "phase2a/summary.json",
    f"{PHASE2B_OUT_NAME}/summary.json",
    "phase1c/summary.json",
    "terminal_logs/phase1c_crosscheck_command.txt",
]
print("Kiem tra day du file bat buoc:")
for rel in expected:
    p = SR / rel
    if p.exists():
        status = "OK"
    elif rel.startswith(PHASE2B_OUT_NAME) and SESSION_MAX_FILES is not None:
        status = "DANG CHO (phien nay chi chay mot phan, binh thuong)"
    else:
        status = "THIEU !!!"
    print(f"  [{status}] {rel}")

Kiem tra day du file bat buoc:
  [OK] environment_kaggle_subset10pct.txt
  [OK] phase1d/summary.json
  [OK] phase1er/summary.json
  [OK] phase2a/summary.json
  [OK] phase2b_even10pct_seed42/summary.json
  [OK] phase1c/summary.json
  [OK] terminal_logs/phase1c_crosscheck_command.txt


In [23]:
# 21. Nen toan bo de tai ve
suffix = "full" if SESSION_MAX_FILES is None else f"first{SESSION_MAX_FILES}"
get_ipython().system(
    f"zip -r /kaggle/working/student_B_subset10pct_bundle_{suffix}.zip /kaggle/working/student_runs"
)
print(f"San sang tai xuong: /kaggle/working/student_B_subset10pct_bundle_{suffix}.zip")

  adding: kaggle/working/student_runs/ (stored 0%)
  adding: kaggle/working/student_runs/student_B/ (stored 0%)
  adding: kaggle/working/student_runs/student_B/phase2a/ (stored 0%)
  adding: kaggle/working/student_runs/student_B/phase2a/case_results.csv (deflated 49%)
  adding: kaggle/working/student_runs/student_B/phase2a/horizon_audit.json (deflated 75%)
  adding: kaggle/working/student_runs/student_B/phase2a/figures/ (stored 0%)
  adding: kaggle/working/student_runs/student_B/phase2a/figures/2A_D1_geometry_transfer.png (deflated 19%)
  adding: kaggle/working/student_runs/student_B/phase2a/figures/2A_D3_horizon_transfer.png (deflated 18%)
  adding: kaggle/working/student_runs/student_B/phase2a/figures/2A_D4_online_trajectories.png (deflated 10%)
  adding: kaggle/working/student_runs/student_B/phase2a/figures/2A_D2_edge_tracking.png (deflated 18%)
  adding: kaggle/working/student_runs/student_B/phase2a/case_results.json (deflated 71%)
  adding: kaggle/working/student_runs/student_B/ph